# 03. Feature engineering và train/test split

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/ml_scratch').is_dir():
    if ROOT == ROOT.parent: raise FileNotFoundError('Mở notebook trong dự án')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import display, Image


## Mục đích
mục đích: Làm sao biến `clean_text` của email ham/spam thành ma trận số mà test không lọt vào từ vựng?

 Các ô dưới chạy trên `clean_emails.csv`.

1. Đọc email đã làm sạch và xem một ham, một spam.
2. `stratified_split`: trong từng lớp, xáo với seed 42, lấy `round(0,2 * số dòng)` làm test. Phần còn lại là train.
3. `TfidfVectorizer` tự viết: bỏ từ ngắn tiếng Anh và từ một ký tự, giữ 1.000 từ xuất hiện nhiều nhất trên train. `idf = log((1+N)/(1+df)) + 1` nằm trong `smooth_idf`. Nhân số lần xuất hiện với IDF, rồi chia chuẩn L2 từng dòng.
4. Test chỉ `transform`. In bảng count / IDF / TF-IDF của một email ham và một email spam thật.

In [ ]:
from ml_scratch.data import load_clean, stratified_split, TfidfVectorizer, save_features
from ml_scratch.evaluation import visualize_features, email_terms

data = load_clean()
print('Clean', len(data), data.label.value_counts().to_dict())
sample = data.groupby('label', as_index=False).head(1)[['label', 'clean_text']].copy()
sample['clean_text'] = sample.clean_text.str.slice(0, 220)
display(sample)

train, test = stratified_split(data)
print('Train', len(train), train.label.value_counts().to_dict())
print('Test', len(test), test.label.value_counts().to_dict())
print('Train/test overlap', len(set(train.clean_text) & set(test.clean_text)))

vectorizer = TfidfVectorizer(max_features=1000)
X_train = vectorizer.fit_transform(train.clean_text)
X_test = vectorizer.transform(test.clean_text)
print('Shapes', X_train.shape, X_test.shape)

for label in ['ham', 'spam']:
    position = test.label.to_numpy().tolist().index(label)
    row = test.iloc[position]
    print(label, 'source_row', row.name)
    print(row.clean_text[:240])
    display(email_terms(vectorizer, row.clean_text, X_test[position]))

save_features(train, test, X_train, X_test, vectorizer)
visualize_features(train, test, vectorizer)

In [ ]:
display(Image(filename=str(ROOT/"outputs/figures/training_testing.png")))
display(Image(filename=str(ROOT/"outputs/figures/feature_idf.png")))

## Đọc số và handoff
Số dòng train/test và shape in ở ô trên. Overlap của `clean_text` phải bằng 0. `training_testing.png` là số ham/spam sau khi chia theo lớp. `feature_idf.png` là 15 từ có IDF cao nhất trên train: từ hiếm, không phải hệ số của KNN hay Naive Bayes.

Bảng `email_terms` của một ham và một spam cho thấy cùng một công thức trên thư thật: `count` là số lần từ trong thư, `idf` học từ train, `tfidf` là count nhân idf rồi chia chuẩn L2 của cả vector. Từ không có trong 1.000 từ train bị bỏ.

Output, cùng thứ tự dòng giữa CSV và NPZ:

- `data/processed/train.csv`, `test.csv`
- `X_train.npz`, `X_test.npz`
- `vocabulary.csv` và `vectorizer.json`

Bước 04 và 05 gọi `load_features()` rồi áp KNN hoặc Naive Bayes lên đúng các vector này.